# RSNA Knee L2 five-fold TPU training

Runs five matching round-2 ConvNeXt warm starts for four fixed epochs using the final complete private L2 label generation, pinned by manifest, label, weight and row-provenance hashes. This launch is unvalidated; it aborts on unsupported runtime or changed inputs. The queued single-device probe does not establish distributed collectives, real-loader throughput, full training completion or a score. CPU transforms and static all-slot encoding preserve the input/model head but change optimization: bfloat16, replica-dependent global batch, fixed within-epoch learning rate, adjusted EMA and global confidence-weighted loss. Gold58 is excluded from training and remains a tuning diagnostic. Each complete epoch commits an EMA; only final validation creates a completion receipt.


In [ ]:
KNEE_SOURCE='"""Shared dataset and model code for training and inference."""\nimport math\n\nimport numpy as np\nimport timm\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nLABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",\n          "PF OA", "Effusion", "Synovitis", "Baker\'s", "Contusion", "Fracture"]\nPLANES = ["Sagittal", "Coronal", "Axial"]\n# slot = plane index + 3 * (1 - fat_suppressed): fat-suppressed planes first\nN_SLOTS = 6\nMEAN = (0.485, 0.456, 0.406)\nSTD = (0.229, 0.224, 0.225)\n\n\ndef slot_of(plane, fat_sup):\n    return PLANES.index(plane) + 3 * (1 - int(fat_sup))\n\n\ndef build_study_table(series_df, n_slices):\n    """series_df: StudyInstanceUID, SeriesInstanceUID, Fat_Suppression, Anatomical_Plane.\n    n_slices: dict series -> cached slice count. Returns {study: [[series ids of slot 0], ...]}\n    with each slot\'s candidates sorted by slice count (largest first)."""\n    table = {}\n    for st, sr, fs, pl in zip(series_df.StudyInstanceUID, series_df.SeriesInstanceUID,\n                              series_df.Fat_Suppression, series_df.Anatomical_Plane):\n        if n_slices.get(sr, 0) < 3:\n            continue\n        table.setdefault(st, [[] for _ in range(N_SLOTS)])[slot_of(pl, fs)].append(sr)\n    for slots in table.values():\n        for cands in slots:\n            cands.sort(key=lambda s: -n_slices[s])\n    return table\n\n\ndef window_centres(n, k, train, lo=0.04, hi=0.96):\n    """k slice indices (centres of 3-slice windows) spread over the series."""\n    a, b = max(1.0, lo * (n - 1)), min(n - 2.0, hi * (n - 1))\n    if b < a:\n        a = b = (n - 1) / 2\n    if train:\n        edges = np.linspace(a, b, k + 1)\n        c = edges[:-1] + np.random.rand(k) * (edges[1:] - edges[:-1])\n    else:\n        c = np.linspace(a, b, k)\n    return np.clip(np.round(c).astype(int), 1, max(1, n - 2))\n\n\nclass StudyDataset(torch.utils.data.Dataset):\n    """One item = one study: uint8 windows [N_SLOTS, k, 3, H, W], slot mask, slice positions, target."""\n\n    def __init__(self, studies, table, cache_dir, targets=None, k=8, train=False, loader=None):\n        self.studies, self.table, self.cache_dir = list(studies), table, cache_dir\n        self.targets, self.k, self.train = targets, k, train\n        self.loader = loader or (lambda s: np.load(f"{self.cache_dir}/{s}.npy", mmap_mode="r"))\n\n    def __len__(self):\n        return len(self.studies)\n\n    def __getitem__(self, i):\n        st = self.studies[i]\n        slots = self.table.get(st, [[] for _ in range(N_SLOTS)])\n        x, mask, pos = None, np.zeros(N_SLOTS, bool), np.zeros((N_SLOTS, self.k), np.float32)\n        for s, cands in enumerate(slots):\n            if not cands:\n                continue\n            sr = cands[np.random.randint(len(cands))] if self.train else cands[0]\n            vol = self.loader(sr)\n            n = len(vol)\n            c = window_centres(n, self.k, self.train)\n            win = np.stack([vol[np.clip([j - 1, j, j + 1], 0, n - 1)] for j in c])  # [k, 3, H, W]\n            if x is None:\n                x = np.zeros((N_SLOTS, self.k) + win.shape[1:], np.uint8)\n            x[s], mask[s], pos[s] = win, True, c / max(n - 1, 1)\n        if x is None:\n            x = np.zeros((N_SLOTS, self.k, 3, 384, 384), np.uint8)\n        y = self.targets[i] if self.targets is not None else np.zeros(len(LABELS), np.float32)\n        return torch.from_numpy(x), torch.from_numpy(mask), torch.from_numpy(pos), torch.from_numpy(np.asarray(y, np.float32))\n\n\ndef make_views(x, res, train, group, fov=0.92):\n    """x: uint8 [N, 3, H, W] with N = n_series * group. Crops a centred `fov` fraction of the cached\n    field of view and resizes to res; in training adds rotation/scale/shift and gain/gamma jitter\n    that is shared within a series."""\n    n = x.shape[0]\n    ns = n // group\n    dev = x.device\n    x = x.float().div_(255.0)\n    if train:\n        ang = (torch.rand(ns, device=dev) - 0.5) * 2 * math.radians(12)\n        sc = fov * (1 + (torch.rand(ns, device=dev) - 0.5) * 0.24)\n        tx = (torch.rand(ns, device=dev) - 0.5) * 0.12\n        ty = (torch.rand(ns, device=dev) - 0.5) * 0.12\n        gamma = torch.exp((torch.rand(ns, device=dev) - 0.5) * 0.5)\n        gain = 1 + (torch.rand(ns, device=dev) - 0.5) * 0.2\n    else:\n        ang = torch.zeros(ns, device=dev)\n        sc = torch.full((ns,), fov, device=dev)\n        tx = ty = torch.zeros(ns, device=dev)\n        gamma = gain = None\n    cos, sin = torch.cos(ang) * sc, torch.sin(ang) * sc\n    theta = torch.stack([torch.stack([cos, -sin, tx], 1), torch.stack([sin, cos, ty], 1)], 1)\n    theta = theta.repeat_interleave(group, 0)\n    grid = F.affine_grid(theta, (n, 3, res, res), align_corners=False)\n    if x.shape[-1] > 1.5 * res:  # avoid aliasing when shrinking a lot\n        x = F.avg_pool2d(x, 2) if x.shape[-1] >= 2 * res else F.interpolate(x, scale_factor=0.75, mode="bilinear", antialias=True)\n    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)\n    if train:\n        g = gamma.repeat_interleave(group).view(n, 1, 1, 1)\n        x = x.clamp_min(1e-4).pow(g) * gain.repeat_interleave(group).view(n, 1, 1, 1)\n    mean = torch.tensor(MEAN, device=dev).view(1, 3, 1, 1)\n    std = torch.tensor(STD, device=dev).view(1, 3, 1, 1)\n    return (x - mean) / std\n\n\nclass KneeNet(nn.Module):\n    """2D backbone on 3-slice windows -> tokens tagged with series slot and slice position ->\n    small transformer over all windows of the study -> one attention pooling per finding."""\n\n    def __init__(self, backbone="convnext_tiny", pretrained=True, dim=384, depth=2, drop=0.2, drop_path=0.1):\n        super().__init__()\n        kw = {"drop_path_rate": drop_path} if drop_path else {}\n        self.enc = timm.create_model(backbone, pretrained=pretrained, num_classes=0, **kw)\n        self.proj = nn.Sequential(nn.Dropout(drop), nn.Linear(self.enc.num_features, dim), nn.LayerNorm(dim))\n        self.slot_emb = nn.Embedding(N_SLOTS, dim)\n        self.pos_emb = nn.Sequential(nn.Linear(1, dim), nn.GELU(), nn.Linear(dim, dim))\n        layer = nn.TransformerEncoderLayer(dim, 8, dim * 2, dropout=0.1, batch_first=True, norm_first=True)\n        self.tf = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False) if depth else None\n        self.att = nn.Sequential(nn.Linear(dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, len(LABELS)))\n        self.cls_w = nn.Parameter(torch.randn(len(LABELS), dim) * 0.02)\n        self.cls_b = nn.Parameter(torch.zeros(len(LABELS)))\n\n    def head(self, feat, mask, pos):\n        """feat [B, S, K, D], mask [B, S] bool, pos [B, S, K] in 0..1 -> logits [B, 12]."""\n        b, s, k, _ = feat.shape\n        slot = torch.arange(s, device=feat.device).view(1, s, 1).expand(b, s, k)\n        h = self.proj(feat) + self.slot_emb(slot) + self.pos_emb(pos.unsqueeze(-1).to(feat.dtype))\n        h = h.view(b, s * k, -1)\n        pad = ~mask.unsqueeze(-1).expand(b, s, k).reshape(b, s * k)\n        pad = pad & ~pad.all(1, keepdim=True)  # a study with no series at all attends to everything\n        if self.tf is not None:\n            h = self.tf(h, src_key_padding_mask=pad)\n        a = self.att(h).float().masked_fill(pad.unsqueeze(-1), -1e4).softmax(1)  # [B, T, C]\n        pooled = torch.einsum("btc,btd->bcd", a.to(h.dtype), h)\n        return (pooled * self.cls_w).sum(-1) + self.cls_b\n\n    def forward(self, x, mask, pos, res=256, train=False):\n        """x uint8 [B, S, K, 3, H, W]."""\n        b, s, k = x.shape[:3]\n        keep = mask.view(-1)  # run the backbone only on series that exist\n        xv = x.view(b * s, k, *x.shape[3:])[keep].flatten(0, 1)\n        f = self.enc(make_views(xv, res, train, k).contiguous(memory_format=torch.channels_last))\n        feat = f.new_zeros(b * s, k, f.shape[-1])\n        feat[keep] = f.view(-1, k, f.shape[-1])\n        return self.head(feat.view(b, s, k, -1), mask, pos)\n'
HELPER_SOURCE='"""Warm-start recovered round-2 ConvNeXt folds using explicit L2 targets.\n\nThe 256px JPEG-q95 study cache and KneeNet implementation are the recovered r2\nones. Gold58 is excluded from training, but has already informed upstream labels\nand model choices; its AUC is a diagnostic, not an independent validation score.\n"""\n\nimport argparse\nimport copy\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport sys\nimport time\n\nos.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import roc_auc_score\nimport torch\nimport torch.distributed as dist\nimport torch.nn.functional as F\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as source:\n        for block in iter(lambda: source.read(8 * 1024 * 1024), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef read_indexed(path):\n    frame = pd.read_csv(path, dtype={"StudyInstanceUID": str})\n    if "StudyInstanceUID" not in frame:\n        raise ValueError(f"{path}: missing StudyInstanceUID")\n    if frame.StudyInstanceUID.isna().any() or frame.StudyInstanceUID.duplicated().any():\n        raise ValueError(f"{path}: missing or duplicate study IDs")\n    return frame.set_index("StudyInstanceUID")\n\n\ndef checked_targets(frame, labels, ids, name):\n    if not set(labels).issubset(frame.columns):\n        raise ValueError(f"{name}: expected all 12 target columns")\n    if set(frame.index) != set(ids):\n        missing = len(set(ids) - set(frame.index))\n        extra = len(set(frame.index) - set(ids))\n        raise ValueError(f"{name}: study ID mismatch; missing {missing}, extra {extra}")\n    value = frame.loc[ids, labels].apply(pd.to_numeric, errors="raise").astype(np.float32)\n    if not np.isfinite(value.to_numpy()).all():\n        raise ValueError(f"{name}: non-finite values")\n    if not ((value.to_numpy() >= 0) & (value.to_numpy() <= 1)).all():\n        raise ValueError(f"{name}: values outside [0, 1]")\n    return value\n\n\ndef load_training_tables(a1, l2, competition, labels, fold):\n    lab = read_indexed(Path(a1) / "labels_cons.csv")\n    official = read_indexed(Path(competition) / "train.csv")\n    ids = official.index.tolist()\n    if set(lab.index) != set(ids):\n        raise ValueError("A1 labels_cons and official train study IDs differ")\n    lab = lab.loc[ids]\n    official_gold = official[labels].notna().all(axis=1)\n    if not np.array_equal(official_gold.to_numpy(), lab.is_gold.to_numpy().astype(bool)):\n        raise ValueError("A1 gold flags differ from the current official labels")\n    folds = pd.to_numeric(lab.fold, errors="raise").astype(int)\n    if set(folds[~official_gold].unique()) != set(range(5)):\n        raise ValueError("A1 must have five existing weak-study folds")\n    if not (folds[official_gold] == -1).all():\n        raise ValueError("Gold studies must have A1 fold=-1")\n    target = checked_targets(read_indexed(Path(l2) / "labels_l2.csv"), labels, ids, "L2 labels")\n    confidence = checked_targets(read_indexed(Path(l2) / "weights_l2.csv"), labels, ids, "L2 confidence")\n    cache_meta = read_indexed(Path(a1) / "cache256_meta.csv")\n    cached = set(cache_meta.index[cache_meta.n_slots > 0])\n    weak = lab[~official_gold & lab.index.isin(cached)]\n    train = weak[(weak.fold != fold) & (confidence.loc[weak.index].sum(axis=1) > 0)]\n    val = weak[weak.fold == fold]\n    gold = lab[official_gold & lab.index.isin(cached)]\n    if not len(train) or not len(val) or not len(gold):\n        raise ValueError("Empty train/val/gold partition")\n    if set(train.index) & (set(val.index) | set(gold.index)):\n        raise ValueError("Train partition overlaps validation or gold")\n    gold_y = official.loc[gold.index, labels].to_numpy(dtype=np.int64)\n    return lab, target, confidence, train, val, gold, gold_y\n\n\ndef macro(y, probabilities, labels):\n    result = {}\n    for i, name in enumerate(labels):\n        if np.unique(y[:, i]).size == 2:\n            result[name] = float(roc_auc_score(y[:, i], probabilities[:, i]))\n    if not result:\n        raise ValueError("No evaluable target has both classes")\n    return float(np.mean(list(result.values()))), result\n\n\ndef main():\n    process_started = time.monotonic()\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--a1", required=True)\n    parser.add_argument("--l2", required=True)\n    parser.add_argument("--competition", required=True)\n    parser.add_argument("--src", required=True)\n    parser.add_argument("--init", required=True)\n    parser.add_argument("--fold", type=int, required=True, choices=range(5))\n    parser.add_argument("--epochs", type=int, default=4)\n    parser.add_argument("--k-train", type=int, default=8, choices=range(1, 17))\n    parser.add_argument("--lr", type=float, default=3e-5)\n    parser.add_argument("--lr-head", type=float, default=1e-4)\n    parser.add_argument("--budget-seconds", type=float, default=0,\n                        help="0 means unlimited. Stop normally with the last completed epoch when budget is tight.")\n    parser.add_argument("--final-eval-reserve", type=float, default=160)\n    parser.add_argument("--output", default="/kaggle/working")\n    args = parser.parse_args()\n    if args.epochs < 1:\n        raise ValueError("epochs must be positive")\n    deadline = process_started + args.budget_seconds if args.budget_seconds > 0 else float("inf")\n    sys.path.insert(0, args.src)\n    from knee import LABELS, N_SLOTS, KneeNet, window_centres\n\n    res, k_train, k_eval, batch_size = 256, args.k_train, 12, 2\n    backbone, weight_decay, ema_decay = "convnext_tiny", 0.02, 0.998\n    ddp = int(os.environ.get("WORLD_SIZE", "1")) > 1\n    if ddp:\n        dist.init_process_group("nccl")\n    rank = dist.get_rank() if ddp else 0\n    local_rank = int(os.environ.get("LOCAL_RANK", "0"))\n    torch.cuda.set_device(local_rank)\n    torch.manual_seed(args.fold * 100 + rank)\n    np.random.seed(args.fold * 100 + rank)\n    cv2.setNumThreads(1)\n    say = (lambda message: print(message, flush=True)) if rank == 0 else (lambda message: None)\n    out_dir = Path(args.output)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    lab, target, confidence, train, val, gold, gold_y = load_training_tables(\n        args.a1, args.l2, args.competition, LABELS, args.fold\n    )\n    cache = Path(args.a1) / "cache256"\n    say(f"L2 fold {args.fold}: train {len(train)} val {len(val)} gold {len(gold)}; confidence weighted BCE")\n    say("Gold AUC is training-excluded but tuning-exposed; weak val is not clean nested distillation OOF.")\n\n    class StudyDataset(torch.utils.data.Dataset):\n        # Same JPEG cache decoding, slots and window sampling as recovered r2.\n        def __init__(self, ids, y, weights, k, training):\n            self.ids, self.y, self.weights = list(ids), y, weights\n            self.k, self.training = k, training\n\n        def __len__(self):\n            return len(self.ids)\n\n        def __getitem__(self, index):\n            study = self.ids[index]\n            x = np.zeros((N_SLOTS, self.k, 3, res, res), np.uint8)\n            mask = np.zeros(N_SLOTS, bool)\n            pos = np.zeros((N_SLOTS, self.k), np.float32)\n            with np.load(cache / f"{study}.npz", allow_pickle=False) as archive:\n                for slot in range(N_SLOTS):\n                    if f"s{slot}_o" not in archive.files:\n                        continue\n                    data, offsets = archive[f"s{slot}_d"], archive[f"s{slot}_o"]\n                    n = len(offsets) - 1\n                    if n < 3:\n                        continue\n                    centers = window_centres(n, self.k, self.training)\n                    decoded = {}\n                    required = sorted({int(t) for center in centers for t in np.clip([center - 1, center, center + 1], 0, n - 1)})\n                    for j in required:\n                        image = cv2.imdecode(data[offsets[j]:offsets[j + 1]], cv2.IMREAD_GRAYSCALE)\n                        if image is None or image.shape != (res, res):\n                            raise ValueError(f"{study} slot {slot} slice {j}: invalid JPEG cache")\n                        decoded[j] = image\n                    x[slot] = np.stack([\n                        np.stack([decoded[int(t)] for t in np.clip([center - 1, center, center + 1], 0, n - 1)])\n                        for center in centers\n                    ])\n                    mask[slot], pos[slot] = True, centers / max(n - 1, 1)\n            if not mask.any():\n                raise ValueError(f"{study}: cache metadata claimed nonempty study but no slot is usable")\n            y = self.y[index] if self.y is not None else np.zeros(len(LABELS), np.float32)\n            w = self.weights[index] if self.weights is not None else np.ones(len(LABELS), np.float32)\n            return tuple(torch.from_numpy(value) for value in (x, mask, pos, np.asarray(y, np.float32), np.asarray(w, np.float32)))\n\n    training = StudyDataset(train.index, target.loc[train.index, LABELS].to_numpy(),\n                            confidence.loc[train.index, LABELS].to_numpy(), k_train, True)\n    sampler = torch.utils.data.distributed.DistributedSampler(training, shuffle=True, drop_last=True) if ddp else None\n    train_loader = torch.utils.data.DataLoader(\n        training, batch_size=batch_size, shuffle=sampler is None, sampler=sampler, drop_last=True,\n        num_workers=2, pin_memory=True, persistent_workers=True, prefetch_factor=3,\n    )\n    initial = torch.load(args.init, map_location="cpu", weights_only=False)\n    saved_args = initial.get("args", {})\n    if saved_args.get("backbone") != backbone or int(saved_args.get("fold", -1)) != args.fold:\n        raise ValueError("Warm-start checkpoint is not the matching r2 ConvNeXt fold")\n    if saved_args.get("mode") != "r2" or int(saved_args.get("res", -1)) != res:\n        raise ValueError("Warm-start checkpoint mode/resolution differ from recovered r2")\n    model = KneeNet(backbone, pretrained=False).cuda().to(memory_format=torch.channels_last)\n    model.load_state_dict(initial["model"], strict=True)\n    del initial\n    ema = copy.deepcopy(model).eval()\n    for param in ema.parameters():\n        param.requires_grad_(False)\n    encoder = list(model.enc.parameters())\n    head = [param for name, param in model.named_parameters() if not name.startswith("enc.")]\n    optimizer = torch.optim.AdamW([{"params": encoder, "lr": args.lr}, {"params": head, "lr": args.lr_head}], weight_decay=weight_decay)\n    network = torch.nn.parallel.DistributedDataParallel(model, device_ids=[local_rank]) if ddp else model\n    steps, warm = args.epochs * len(train_loader), max(1, len(train_loader) // 4)\n    scheduler = torch.optim.lr_scheduler.LambdaLR(\n        optimizer, lambda step: min(1.0, (step + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, step / max(1, steps))))\n    )\n    scaler = torch.cuda.amp.GradScaler()\n\n    @torch.no_grad()\n    def predict(net, ids):\n        loader = torch.utils.data.DataLoader(StudyDataset(ids, None, None, k_eval, False),\n                                            batch_size=4, num_workers=2, pin_memory=True)\n        predictions = []\n        net.eval()\n        for x, mask, pos, _, _ in loader:\n            with torch.autocast("cuda", dtype=torch.float16):\n                probability = net(x.cuda(non_blocking=True), mask.cuda(), pos.cuda(), res=res).float().sigmoid()\n            if not torch.isfinite(probability).all():\n                raise ValueError("Non-finite L2 validation predictions")\n            predictions.append(probability.cpu())\n        return torch.cat(predictions).numpy()\n\n    hashes = {\n        "labels_l2.csv": digest(Path(args.l2) / "labels_l2.csv"),\n        "weights_l2.csv": digest(Path(args.l2) / "weights_l2.csv"),\n        "labels_cons.csv": digest(Path(args.a1) / "labels_cons.csv"),\n        "knee.py": digest(Path(args.src) / "knee.py"),\n        "warm_start": digest(args.init),\n        "trainer": digest(__file__),\n    }\n    init_gold = None\n    history, started, step = [], time.time(), 0\n    completed_epochs, prior_training_seconds = 0, None\n    checkpoint_path = out_dir / f"student_l2_convnext_tiny_f{args.fold}.pt"\n    if rank == 0:\n        init_gold = predict(ema, gold.index)\n        value, per_target = macro(gold_y, init_gold, LABELS)\n        say(f"INIT r2 fold {args.fold}: gold diagnostic {value:.5f}")\n        pd.DataFrame(init_gold, index=pd.Index(gold.index, name="StudyInstanceUID"), columns=LABELS).to_csv(out_dir / f"init_r2_gold_f{args.fold}.csv")\n    if ddp:\n        dist.barrier()\n    for epoch in range(args.epochs):\n        cannot_complete = int(bool(completed_epochs) and time.monotonic() + prior_training_seconds + 35 > deadline)\n        stop_epoch = torch.tensor([cannot_complete], device="cuda")\n        if ddp:\n            dist.all_reduce(stop_epoch, op=dist.ReduceOp.MAX)\n        if bool(stop_epoch.item()):\n            say("L2_TIME_GUARD: insufficient time for another completed epoch; retaining last completed EMA")\n            break\n        if sampler is not None:\n            sampler.set_epoch(epoch)\n        network.train()\n        total, count, epoch_start = 0.0, 0, time.time()\n        partial_epoch = False\n        for x, mask, pos, y, weights in train_loader:\n            stop = torch.tensor([int(time.monotonic() + 35 > deadline)], device="cuda")\n            if ddp:\n                dist.all_reduce(stop, op=dist.ReduceOp.MAX)\n            if bool(stop.item()):\n                partial_epoch = True\n                break\n            x, mask, pos = x.cuda(non_blocking=True), mask.cuda(), pos.cuda()\n            y, weights = y.cuda(), weights.cuda()\n            with torch.autocast("cuda", dtype=torch.float16):\n                logits = network(x, mask, pos, res=res, train=True)\n            per_target_loss = F.binary_cross_entropy_with_logits(logits.float(), y, reduction="none")\n            loss = (per_target_loss * weights).sum() / weights.sum().clamp_min(1e-8)\n            if not torch.isfinite(loss):\n                raise ValueError("Non-finite confidence-weighted training loss")\n            optimizer.zero_grad(set_to_none=True)\n            scaler.scale(loss).backward()\n            scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)\n            scaler.step(optimizer)\n            scaler.update()\n            scheduler.step()\n            step += 1\n            # A pretrained warm start should not reset EMA to the first few batches.\n            with torch.no_grad():\n                for averaged, current in zip(ema.parameters(), model.parameters()):\n                    averaged.lerp_(current, 1 - ema_decay)\n                for averaged, current in zip(ema.buffers(), model.buffers()):\n                    averaged.copy_(current)\n            total += loss.item()\n            count += 1\n            if rank == 0 and count % 100 == 0:\n                say(f"ep {epoch} {count}/{len(train_loader)} loss {total/count:.4f} {(time.time()-epoch_start)/count:.2f}s/it")\n        if partial_epoch:\n            if completed_epochs == 0:\n                raise RuntimeError("L2_TIME_GUARD: no full L2 epoch completed; no trained L2 checkpoint is available")\n            saved = torch.load(checkpoint_path, map_location="cpu", weights_only=False)\n            ema.load_state_dict(saved["model"], strict=True)\n            del saved\n            say("L2_TIME_GUARD: discarded partial epoch and restored last completed EMA")\n            if ddp:\n                dist.barrier()\n            break\n        prior_training_seconds = time.time() - epoch_start\n        completed_epochs += 1\n        if rank == 0:\n            pg = predict(ema, gold.index)\n            value, per_target = macro(gold_y, pg, LABELS)\n            item = {"epoch": epoch, "loss": total / max(1, count), "gold_diagnostic_auc": value,\n                    "gold_diagnostic_per_target": per_target, "seconds": time.time() - epoch_start}\n            history.append(item)\n            say(f"ep {epoch}: loss {item[\'loss\']:.4f}; gold diagnostic {value:.5f}; elapsed {time.time()-started:.0f}s")\n            provenance = {\n                "targets": LABELS, "source_hashes": hashes, "fold": args.fold,\n                "train_ids_sha256": hashlib.sha256("\\n".join(train.index).encode()).hexdigest(),\n                "val_ids_sha256": hashlib.sha256("\\n".join(val.index).encode()).hexdigest(),\n                "gold_excluded_from_gradient_training": True,\n                "gold_auc_independent_validation": False,\n                "weak_val_independent_of_label_generation": False,\n                "selection": "last fixed epoch; no gold-best selection",\n                "preprocessing": "A1 cached32 slices; area256; JPEGq95; real six slots without FS fallback",\n                "source_knee": "recovered r2 ConvNeXt kernel output src/knee.py",\n                "completed_epochs": completed_epochs,\n                "requested_epochs": args.epochs,\n                "label_source": "private L2 labels_l2.csv, confidence weights_l2.csv",\n                "fold_source": "A1 labels_cons.csv; preserved weak-study report-group folds",\n                "gold_source": "current official train.csv; all completely annotated studies excluded",\n            }\n            checkpoint = {"model": ema.state_dict(), "args": {\n                "backbone": backbone, "res": res, "k_eval": k_eval, "k_train": k_train,\n                "mode": "l2", "fold": args.fold, "epoch": epoch, "epochs": args.epochs,\n                "lr": args.lr, "lr_head": args.lr_head, "dim": 384, "depth": 2,\n                "drop": 0.2, "drop_path": 0.1, "confidence_weighted_bce": True,\n            }, "provenance": provenance}\n            path = checkpoint_path\n            temporary = path.with_suffix(".pt.tmp")\n            torch.save(checkpoint, temporary)\n            os.replace(temporary, path)\n            (out_dir / f"training_l2_f{args.fold}.json").write_text(json.dumps({"args": vars(args), "history": history, "provenance": provenance}, indent=2))\n        if ddp:\n            dist.barrier()\n    if rank == 0:\n        frames = [pd.DataFrame(pg, index=pd.Index(gold.index, name="StudyInstanceUID"), columns=LABELS).assign(set="gold")]\n        full_val_saved = False\n        if time.monotonic() + args.final_eval_reserve < deadline:\n            pv = predict(ema, val.index)\n            vy = (lab.loc[val.index, LABELS].to_numpy() > 0.5).astype(np.int64)\n            val_auc, _ = macro(vy, pv, LABELS)\n            frames.insert(0, pd.DataFrame(pv, index=pd.Index(val.index, name="StudyInstanceUID"), columns=LABELS).assign(set="val"))\n            full_val_saved = True\n            say(f"FINAL L2 fold {args.fold}: weak consensus diagnostic {val_auc:.5f}")\n        else:\n            say("L2_TIME_GUARD: full weak-val prediction omitted; completed checkpoint and gold diagnostics retained")\n        gold_auc, gold_per_target = macro(gold_y, pg, LABELS)\n        say(f"FINAL L2 fold {args.fold}: completed epochs {completed_epochs}; gold diagnostic {gold_auc:.5f}")\n        say(json.dumps(gold_per_target))\n        frame = pd.concat(frames)\n        frame.to_csv(out_dir / f"oof_l2_convnext_tiny_f{args.fold}.csv")\n        (out_dir / f"completion_l2_f{args.fold}.json").write_text(json.dumps({\n            "fold": args.fold, "completed_epochs": completed_epochs, "requested_epochs": args.epochs,\n            "k_train": k_train, "k_eval": k_eval,\n            "full_weak_val_saved": full_val_saved, "gold_diagnostic_auc": gold_auc,\n            "checkpoint": checkpoint_path.name, "checkpoint_sha256": digest(checkpoint_path),\n            "seconds": time.monotonic() - process_started,\n        }, indent=2))\n    if ddp:\n        dist.destroy_process_group()\n\n\nif __name__ == "__main__":\n    main()\n'
TPU_SOURCE='"""Distributed TPU warm-start of all five L2 ConvNeXt-T folds.\n\nThe JPEG cache, targets, model head and checkpoints are the recovered r2 ones.\nViews are rendered on CPU; the static TPU encoder evaluates all six slots and\nthen zeros missing-slot features. ConvNeXt uses LayerNorm rather than BatchNorm.\nGold58 is a tuning diagnostic and never enters the training loader.\n"""\nimport argparse\nimport copy\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport sys\nimport time\n\nos.environ.setdefault(\'PJRT_DEVICE\', \'TPU\')\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn.functional as F\nimport torch_xla\nimport torch_xla.core.xla_model as xm\nimport torch_xla.runtime as xr\nimport torch_xla.debug.metrics as xla_metrics\n\n\nEXPECTED_LABEL_GENERATION = {\n    \'dataset\': \'gwey1025/rsna-knee-l2-labels-20261015\',\n    \'version\': 1,\n    \'annotations_complete\': True,\n    \'annotated_studies\': 4349,\n    \'report_only_studies\': 4349,\n    \'studies\': 4407,\n    \'gold_excluded_from_training\': 58,\n    \'is_pilot\': False,\n    \'algorithm\': \'l2_fixed_round2_report_shares_v1\',\n    \'hashes\': {\n        \'manifest.json\': \'477c5902b982ed8dc67d730fbf55f807df1a40fbed1de38bc1211c980cd4964c\',\n        \'labels_l2.csv\': \'4e28bbc05c08ae4509d0eab264ebb88365a0b2cb0683dcb40b63bce5b9dab315\',\n        \'weights_l2.csv\': \'7d0d5f9a11c28f1ab58bc1dbb4220ea87c34bc75c0104fdc91977353f0db4839\',\n        \'row_provenance.csv\': \'45564b7e483be02da6ab0a0913888ad307d74d7a8acaaf4c77652b3b27ca63fe\',\n    },\n}\n\n\ndef verify_label_generation(folder):\n    folder = Path(folder)\n    for name, expected in EXPECTED_LABEL_GENERATION[\'hashes\'].items():\n        actual = hashlib.sha256((folder / name).read_bytes()).hexdigest()\n        if actual != expected:\n            raise ValueError(\'TPU label generation digest differs: \' + name)\n    manifest = json.loads((folder / \'manifest.json\').read_text())\n    for name in (\'annotations_complete\', \'annotated_studies\', \'report_only_studies\',\n                 \'studies\', \'gold_excluded_from_training\', \'is_pilot\', \'algorithm\'):\n        if manifest.get(name) != EXPECTED_LABEL_GENERATION[name]:\n            raise ValueError(\'TPU requires the final complete label generation: \' + name)\n    for name in (\'labels_l2.csv\', \'weights_l2.csv\', \'row_provenance.csv\'):\n        if manifest[\'outputs\'].get(name) != EXPECTED_LABEL_GENERATION[\'hashes\'][name]:\n            raise ValueError(\'TPU label manifest output digest differs: \' + name)\n    return dict(EXPECTED_LABEL_GENERATION)\n\n\ndef worker(index, args):\n    sys.path.insert(0, args.src)\n    from knee import KneeNet, LABELS, N_SLOTS, window_centres, make_views\n    from train_student_l2 import load_training_tables, macro, digest\n    label_generation = verify_label_generation(args.l2)\n    device = torch_xla.device()\n    rank, world = xr.global_ordinal(), xr.world_size()\n    if xm.xla_device_hw(device) != \'TPU\':\n        raise RuntimeError(\'This notebook requires an actual TPU runtime\')\n    torch.set_num_threads(1)\n    cv2.setNumThreads(1)\n    torch.backends.mha.set_fastpath_enabled(False)\n    output = Path(args.output)\n    output.mkdir(parents=True, exist_ok=True)\n    folds = [int(value) for value in args.folds.split(\',\')]\n    global_batch = args.batch_size * world\n    ema_decay = 0.998 ** (global_batch / 4)\n    say = (lambda message: print(message, flush=True)) if rank == 0 else (lambda message: None)\n    say(f\'TPU world={world} global_batch={global_batch} EMA={ema_decay:.6f}\')\n    receipts = []\n    common_hashes = {\n        \'manifest.json\': digest(Path(args.l2) / \'manifest.json\'),\n        \'row_provenance.csv\': digest(Path(args.l2) / \'row_provenance.csv\'),\n        \'labels_l2.csv\': digest(Path(args.l2) / \'labels_l2.csv\'),\n        \'weights_l2.csv\': digest(Path(args.l2) / \'weights_l2.csv\'),\n        \'labels_cons.csv\': digest(Path(args.a1) / \'labels_cons.csv\'),\n        \'knee.py\': digest(Path(args.src) / \'knee.py\'),\n        \'training_tables_helper\': digest(Path(args.src) / \'train_student_l2.py\'),\n        \'trainer\': digest(__file__),\n    }\n    if common_hashes[\'knee.py\'] != \'314c2af387a5cbd1d25f192f19f4ecc6d4ae4b30403e92e9d9b3ed2c325880b1\':\n        raise ValueError(\'TPU source differs from the recovered r2 knee.py\')\n\n    class Dataset(torch.utils.data.Dataset):\n        def __init__(self, ids, targets, weights, k, training):\n            self.ids, self.targets, self.weights = list(ids), targets, weights\n            self.k, self.training = k, training\n\n        def __len__(self):\n            return len(self.ids)\n\n        def __getitem__(self, position):\n            study = self.ids[position]\n            raw = np.zeros((N_SLOTS, self.k, 3, 256, 256), np.uint8)\n            mask = np.zeros(N_SLOTS, bool)\n            locations = np.zeros((N_SLOTS, self.k), np.float32)\n            with np.load(Path(args.a1) / \'cache256\' / f\'{study}.npz\', allow_pickle=False) as archive:\n                for slot in range(N_SLOTS):\n                    if f\'s{slot}_o\' not in archive.files:\n                        continue\n                    data, offsets = archive[f\'s{slot}_d\'], archive[f\'s{slot}_o\']\n                    n = len(offsets) - 1\n                    if n < 3:\n                        continue\n                    centers = window_centres(n, self.k, self.training)\n                    required = sorted({int(t) for center in centers for t in np.clip([center-1, center, center+1], 0, n-1)})\n                    decoded = {}\n                    for number in required:\n                        image = cv2.imdecode(data[offsets[number]:offsets[number+1]], cv2.IMREAD_GRAYSCALE)\n                        if image is None or image.shape != (256, 256):\n                            raise ValueError(\'Invalid JPEG cache for \' + study)\n                        decoded[number] = image\n                    raw[slot] = np.stack([np.stack([decoded[int(t)] for t in np.clip([center-1, center, center+1], 0, n-1)]) for center in centers])\n                    mask[slot], locations[slot] = True, centers / max(n-1, 1)\n            if not mask.any():\n                raise ValueError(\'Empty cached study: \' + study)\n            # Fixed all-slot shape avoids data-dependent boolean indexing on XLA.\n            views = make_views(torch.from_numpy(raw).flatten(0, 1), 256, self.training, self.k)\n            target = self.targets[position] if self.targets is not None else np.zeros(12, np.float32)\n            weight = self.weights[position] if self.weights is not None else np.ones(12, np.float32)\n            return views, torch.from_numpy(mask), torch.from_numpy(locations), torch.from_numpy(np.asarray(target, np.float32)), torch.from_numpy(np.asarray(weight, np.float32))\n\n    def forward(model, views, mask, locations, k):\n        batch = mask.shape[0]\n        features = model.enc(views.flatten(0, 1)).reshape(batch, N_SLOTS, k, -1)\n        features = features * mask.unsqueeze(-1).unsqueeze(-1)\n        return model.head(features, mask, locations)\n\n    def predict(model, ids, tag):\n        ids = list(ids)\n        per_replica = (len(ids) + world - 1) // world\n        assigned = [ids[position] for position in range(rank, len(ids), world)]\n        actual = len(assigned)\n        assigned += [ids[0]] * (per_replica - actual)\n        dataset = Dataset(assigned, None, None, 12, False)\n        loader = torch.utils.data.DataLoader(dataset, batch_size=1, num_workers=0)\n        predictions = []\n        model.eval()\n        with torch.no_grad():\n            for views, mask, locations, _, _ in loader:\n                with torch.autocast(\'xla\', dtype=torch.bfloat16):\n                    logits = forward(model, views.to(device), mask.to(device), locations.to(device), 12)\n                xm.mark_step()\n                predictions.append(logits.float().sigmoid().cpu().numpy()[0].tolist())\n        chunks = xm.mesh_reduce(tag, (assigned[:actual], predictions[:actual]), lambda pieces: pieces)\n        if rank != 0:\n            return None\n        lookup = {}\n        for names, probabilities in chunks:\n            for study, probability in zip(names, probabilities):\n                if study in lookup:\n                    raise ValueError(\'Duplicate distributed prediction ID\')\n                lookup[study] = probability\n        if set(lookup) != set(ids):\n            raise ValueError(\'Distributed prediction coverage differs from requested IDs\')\n        result = np.asarray([lookup[study] for study in ids], np.float32)\n        if not np.isfinite(result).all():\n            raise ValueError(\'Nonfinite TPU predictions\')\n        return result\n\n    def save_epoch(model, fold, completed, provenance):\n        # Every replica reaches both rendezvous calls; only global rank zero\n        # serializes CPU tensors. Commit an EMA before optional final validation.\n        saved = {\'model\': model.state_dict(), \'args\': {\n            \'backbone\': \'convnext_tiny\', \'res\': 256, \'k_eval\': 12, \'k_train\': 8,\n            \'mode\': \'l2\', \'fold\': fold, \'epoch\': completed - 1, \'epochs\': args.epochs,\n            \'backend\': \'xla\', \'dim\': 384, \'depth\': 2, \'drop\': 0.2, \'drop_path\': 0.1,\n            \'lr\': args.lr, \'lr_head\': args.lr_head, \'confidence_weighted_bce\': True,\n        }, \'provenance\': provenance}\n        path = output / f\'student_l2_convnext_tiny_f{fold}.pt\'\n        temporary = path.with_suffix(\'.pt.tmp\')\n        xm.save(saved, str(temporary), master_only=True, global_master=True)\n        xm.rendezvous(f\'l2-f{fold}-epoch{completed}-written\')\n        if rank == 0:\n            os.replace(temporary, path)\n        xm.rendezvous(f\'l2-f{fold}-epoch{completed}-committed\')\n        return path\n\n    for fold in folds:\n        started = time.monotonic()\n        torch.manual_seed(fold*100 + rank)\n        xm.set_rng_state(fold*100 + rank, device=device)\n        np.random.seed(fold*100 + rank)\n        lab, targets, confidence, training, val, gold, gold_y = load_training_tables(args.a1, args.l2, args.competition, LABELS, fold)\n        dataset = Dataset(training.index, targets.loc[training.index, LABELS].to_numpy(), confidence.loc[training.index, LABELS].to_numpy(), 8, True)\n        sampler = torch.utils.data.distributed.DistributedSampler(dataset, num_replicas=world, rank=rank, shuffle=True, drop_last=True)\n        loader = torch.utils.data.DataLoader(dataset, batch_size=args.batch_size, sampler=sampler, drop_last=True, num_workers=0)\n        if not len(loader):\n            raise ValueError(\'Global TPU batch leaves no full training batch\')\n        checkpoint_path = Path(args.r2) / f\'student_r2_convnext_tiny_f{fold}.pt\'\n        initial = torch.load(checkpoint_path, map_location=\'cpu\', weights_only=False)\n        saved_args = initial.get(\'args\', {})\n        if saved_args.get(\'mode\') != \'r2\' or saved_args.get(\'backbone\') != \'convnext_tiny\' or int(saved_args.get(\'fold\', -1)) != fold or int(saved_args.get(\'res\', -1)) != 256:\n            raise ValueError(\'TPU warm start requires the matching r2 ConvNeXt fold\')\n        model = KneeNet(\'convnext_tiny\', pretrained=False)\n        model.load_state_dict(initial[\'model\'], strict=True)\n        del initial\n        model.to(device)\n        ema = copy.deepcopy(model).eval()\n        for parameter in ema.parameters():\n            parameter.requires_grad_(False)\n        encoder = list(model.enc.parameters())\n        head = [parameter for name, parameter in model.named_parameters() if not name.startswith(\'enc.\')]\n        # XLA supports the single-tensor capturable AdamW path. Its step and\n        # bias corrections remain tensors instead of changing Python constants.\n        optimizer = torch.optim.AdamW([{\'params\': encoder, \'lr\': args.lr}, {\'params\': head, \'lr\': args.lr_head}],\n                                      weight_decay=0.02, capturable=True, foreach=False)\n        hashes = {**common_hashes, \'warm_start\': digest(checkpoint_path)}\n        provenance = {\n            \'targets\': LABELS, \'target_order\': LABELS, \'source_hashes\': hashes,\n            \'fold\': fold, \'backend\': \'xla\',\n            \'train_ids_sha256\': hashlib.sha256(\'\\n\'.join(training.index).encode()).hexdigest(),\n            \'val_ids_sha256\': hashlib.sha256(\'\\n\'.join(val.index).encode()).hexdigest(),\n            \'gold_excluded_from_gradient_training\': True,\n            \'gold_auc_independent_validation\': False,\n            \'gold_auc_is_independent_validation\': False,\n            \'weak_val_independent_of_label_generation\': False,\n            \'selection\': \'last fixed epoch; no gold-best selection\',\n            \'preprocessing\': \'A1 cached32 slices; area256; JPEGq95; real six slots without FS fallback\',\n            \'source_knee\': \'recovered r2 ConvNeXt kernel output src/knee.py\',\n            \'requested_epochs\': args.epochs, \'completed_epochs\': 0,\n            \'label_source\': \'private L2 labels_l2.csv, confidence weights_l2.csv\',\n            \'label_generation\': label_generation,\n            \'fold_source\': \'A1 labels_cons.csv; preserved weak-study report-group folds\',\n            \'gold_source\': \'current official train.csv; all completely annotated studies excluded\',\n            \'loss_reduction\': \'sum confidence-weighted BCE / global replica confidence mass\',\n            \'global_batch\': global_batch, \'ema_decay\': ema_decay,\n            \'precision\': \'bfloat16 autocast with float32 loss and parameters\',\n            \'optimizer\': \'AdamW capturable=True foreach=False; clip after replica gradient reduction\',\n            \'torch_version\': torch.__version__, \'torch_xla_version\': torch_xla.__version__,\n        }\n        history = []\n        for epoch in range(args.epochs):\n            sampler.set_epoch(epoch)\n            model.train()\n            tick = time.monotonic()\n            # Fixed LR per epoch avoids recompiling for every Python scheduler value.\n            decay = 0.5 * (1 + np.cos(np.pi * epoch / max(1, args.epochs)))\n            for group, base_lr in zip(optimizer.param_groups, [args.lr, args.lr_head]):\n                group[\'lr\'] = base_lr * decay\n            total = torch.zeros((), device=device)\n            for step, (views, mask, locations, target, weight) in enumerate(loader):\n                views, mask, locations = views.to(device), mask.to(device), locations.to(device)\n                target, weight = target.to(device), weight.to(device)\n                optimizer.zero_grad(set_to_none=True)\n                with torch.autocast(\'xla\', dtype=torch.bfloat16):\n                    logits = forward(model, views, mask, locations, 8)\n                weight_mass = xm.all_reduce(xm.REDUCE_SUM, weight.sum())\n                numerator = (F.binary_cross_entropy_with_logits(logits.float(), target, reduction=\'none\') * weight).sum()\n                # reduce_gradients averages gradients. Multiplication by world\n                # gives the confidence-weighted objective over the whole batch.\n                loss = numerator * world / weight_mass.clamp_min(1e-8)\n                loss.backward()\n                xm.reduce_gradients(optimizer)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0, foreach=False)\n                optimizer.step()\n                with torch.no_grad():\n                    for shadow, parameter in zip(ema.parameters(), model.parameters()):\n                        shadow.mul_(ema_decay).add_(parameter, alpha=1-ema_decay)\n                    for shadow, buffer in zip(ema.buffers(), model.buffers()):\n                        shadow.copy_(buffer)\n                reduced_loss = xm.all_reduce(xm.REDUCE_SUM, loss.detach(), scale=1.0/world)\n                total += reduced_loss\n                xm.mark_step()\n                if step % 100 == 0:\n                    number = float(reduced_loss.cpu())\n                    if not np.isfinite(number):\n                        raise ValueError(\'Nonfinite L2 TPU training loss\')\n                    say(f\'TPU fold{fold} epoch{epoch} step{step}/{len(loader)} loss={number:.5f} seconds={time.monotonic()-tick:.1f}\')\n            mean_loss = float((total / len(loader)).cpu())\n            if not np.isfinite(mean_loss):\n                raise ValueError(\'Nonfinite completed TPU epoch loss\')\n            history.append({\'epoch\': epoch+1, \'weighted_loss\': mean_loss, \'seconds\': time.monotonic()-tick})\n            say(f\'TPU_EPOCH_COMPLETE fold={fold} epoch={epoch+1} loss={mean_loss:.6f} seconds={time.monotonic()-tick:.1f}\')\n            provenance[\'completed_epochs\'] = epoch + 1\n            path = save_epoch(ema, fold, epoch + 1, provenance)\n            if rank == 0:\n                progress = {\'fold\': fold, \'completed_epochs\': epoch + 1, \'requested_epochs\': args.epochs,\n                            \'checkpoint\': path.name, \'checkpoint_sha256\': digest(path),\n                            \'history\': history, \'provenance\': provenance, \'final_validation_complete\': False}\n                (output / f\'training_l2_f{fold}.json\').write_text(json.dumps(progress, indent=2))\n                (output / f\'xla_metrics_f{fold}.txt\').write_text(xla_metrics.metrics_report())\n            xm.rendezvous(f\'l2-f{fold}-epoch{epoch+1}-progress\')\n        pg = predict(ema, gold.index, f\'l2-f{fold}-gold\')\n        pv = predict(ema, val.index, f\'l2-f{fold}-val\')\n        if rank == 0:\n            gold_auc, per_class = macro(gold_y, pg, LABELS)\n            predictions = pd.concat([pd.DataFrame(pv, index=val.index, columns=LABELS).assign(set=\'val\'),\n                                     pd.DataFrame(pg, index=gold.index, columns=LABELS).assign(set=\'gold\')])\n            predictions.index.name = \'StudyInstanceUID\'\n            predictions.to_csv(output / f\'oof_l2_convnext_tiny_f{fold}.csv\')\n            receipt = {\'fold\': fold, \'completed_epochs\': args.epochs, \'requested_epochs\': args.epochs,\n                       \'backend\': \'xla\', \'world_size\': world,\n                       \'global_batch\': global_batch, \'k_train\': 8, \'k_eval\': 12, \'gold_auc_diagnostic\': gold_auc,\n                       \'gold_auc_is_independent_validation\': False, \'gold_per_class\': per_class,\n                       \'checkpoint_sha256\': digest(path), \'checkpoint\': path.name,\n                       \'elapsed_seconds\': time.monotonic()-started, \'history\': history,\n                       \'target_order\': LABELS, \'gold_excluded_from_gradient_training\': True,\n                       \'full_weak_val_saved\': True, \'source_hashes\': hashes,\n                       \'provenance\': provenance}\n            (output / f\'completion_l2_f{fold}.json\').write_text(json.dumps(receipt, indent=2))\n            receipts.append(receipt)\n            say(\'TPU_L2_FOLD_COMPLETE \' + json.dumps(receipt))\n        xm.rendezvous(f\'l2-f{fold}-receipt\')\n        del model, ema, optimizer, loader, dataset\n        xm.mark_step()\n    if rank == 0:\n        manifest = {\'model_family\': \'l2_convnext_tiny\', \'backend\': \'xla\', \'requested_folds\': folds,\n                    \'requested_epochs\': args.epochs, \'five_folds_complete\': folds == [0,1,2,3,4] and len(receipts)==5,\n                    \'smoke_only\': False, \'targets\': LABELS, \'receipts\': receipts,\n                    \'gold_auc_is_independent_validation\': False,\n                    \'label_generation\': label_generation}\n        (output/\'l2_training_manifest.json\').write_text(json.dumps(manifest, indent=2))\n        say(\'TPU_L2_TRAINING_COMPLETE \' + json.dumps(manifest))\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    for name in [\'a1\',\'l2\',\'r2\',\'competition\',\'src\']:\n        parser.add_argument(\'--\'+name, required=True)\n    parser.add_argument(\'--folds\', default=\'0,1,2,3,4\')\n    parser.add_argument(\'--epochs\', type=int, default=4)\n    parser.add_argument(\'--batch-size\', type=int, default=1)\n    parser.add_argument(\'--lr\', type=float, default=3e-5)\n    parser.add_argument(\'--lr-head\', type=float, default=1e-4)\n    parser.add_argument(\'--output\', default=\'/kaggle/working\')\n    configuration = parser.parse_args()\n    if configuration.epochs < 1:\n        raise ValueError(\'epochs must be positive\')\n    parsed_folds = [int(value) for value in configuration.folds.split(\',\')]\n    if len(set(parsed_folds)) != len(parsed_folds) or not set(parsed_folds).issubset(range(5)):\n        raise ValueError(\'folds must contain unique values between zero and four\')\n    if configuration.batch_size < 1:\n        raise ValueError(\'batch-size must be positive\')\n    torch_xla.launch(worker, args=(configuration,))\n'


In [ ]:
import os,sys,time,json,glob,subprocess,importlib.util,hashlib
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path
def mount(slug, required):
    paths=[Path("/kaggle/input")/slug]
    paths += [Path(p) for p in glob.glob(f"/kaggle/input/**/{slug}",recursive=True)]
    matches=[p for p in paths if (p/required).is_file()]
    if not matches:raise FileNotFoundError(f"{slug}/{required} not attached")
    return matches[0]
A1=mount("rsna-knee-a1-train-cache-labels","labels_cons.csv")
R2=mount("rsna-knee-b-student-f01234-r2-convnext","student_r2_convnext_tiny_f0.pt")
L2=mount("rsna-knee-l2-labels-20261015","labels_l2.csv")
COMP=next((p for p in [Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),Path("/kaggle/input/rsna-knee-abnormality-detection")] if (p/"train.csv").is_file()),None)
if COMP is None:raise FileNotFoundError("Competition train.csv missing")

EXPECTED_LABEL_GENERATION={'dataset': 'gwey1025/rsna-knee-l2-labels-20261015', 'version': 1, 'annotations_complete': True, 'annotated_studies': 4349, 'report_only_studies': 4349, 'studies': 4407, 'gold_excluded_from_training': 58, 'is_pilot': False, 'algorithm': 'l2_fixed_round2_report_shares_v1', 'hashes': {'manifest.json': '477c5902b982ed8dc67d730fbf55f807df1a40fbed1de38bc1211c980cd4964c', 'labels_l2.csv': '4e28bbc05c08ae4509d0eab264ebb88365a0b2cb0683dcb40b63bce5b9dab315', 'weights_l2.csv': '7d0d5f9a11c28f1ab58bc1dbb4220ea87c34bc75c0104fdc91977353f0db4839', 'row_provenance.csv': '45564b7e483be02da6ab0a0913888ad307d74d7a8acaaf4c77652b3b27ca63fe'}}
def verify_label_generation(folder):
    folder = Path(folder)
    for name, expected in EXPECTED_LABEL_GENERATION['hashes'].items():
        actual = hashlib.sha256((folder / name).read_bytes()).hexdigest()
        if actual != expected:
            raise ValueError('TPU label generation digest differs: ' + name)
    manifest = json.loads((folder / 'manifest.json').read_text())
    for name in ('annotations_complete', 'annotated_studies', 'report_only_studies',
                 'studies', 'gold_excluded_from_training', 'is_pilot', 'algorithm'):
        if manifest.get(name) != EXPECTED_LABEL_GENERATION[name]:
            raise ValueError('TPU requires the final complete label generation: ' + name)
    for name in ('labels_l2.csv', 'weights_l2.csv', 'row_provenance.csv'):
        if manifest['outputs'].get(name) != EXPECTED_LABEL_GENERATION['hashes'][name]:
            raise ValueError('TPU label manifest output digest differs: ' + name)
    return dict(EXPECTED_LABEL_GENERATION)
verify_label_generation(L2)
print("TPU_COMPLETE_LABEL_PREFLIGHT",json.dumps(EXPECTED_LABEL_GENERATION),flush=True)

os.environ.setdefault("PJRT_DEVICE","TPU")
if sys.version_info < (3,11):
    raise RuntimeError("Pinned PyTorch/XLA 2.8 requires Python 3.11 or newer")
def package_version(name):
    try:return version(name)
    except PackageNotFoundError:return "missing"
expected={"torch":"2.8.0","torch-xla":"2.8.0","torchvision":"0.23.0","timm":"1.0.22"}
installed={name:package_version(name) for name in expected}
print("TPU_BOOTSTRAP_VERSIONS",installed,flush=True)
if any(installed[name].split("+")[0]!=target for name,target in expected.items()):
    # Match torchvision to torch before timm imports it. CPU wheels avoid an
    # unnecessary CUDA stack in the TPU environment. No torch import precedes pip.
    subprocess.run([sys.executable,"-m","pip","install","torch==2.8.0","torchvision==0.23.0",
                    "--index-url","https://download.pytorch.org/whl/cpu"],check=True)
    subprocess.run([sys.executable,"-m","pip","install","torch_xla[tpu]==2.8.0","timm==1.0.22"],check=True)
missing=[]
if importlib.util.find_spec("cv2") is None:missing.append("opencv-python-headless==4.11.0.86")
if importlib.util.find_spec("sklearn") is None:missing.append("scikit-learn")
if missing:subprocess.run([sys.executable,"-m","pip","install",*missing],check=True)
import torch,torch_xla,timm,torchvision
assert torch.__version__.split("+")[0]=="2.8.0"
assert torch_xla.__version__.split("+")[0]=="2.8.0"
assert torchvision.__version__.split("+")[0]=="0.23.0"
assert timm.__version__=="1.0.22"
print("BACKEND_VERSIONS",torch.__version__,torch_xla.__version__,torchvision.__version__,timm.__version__,flush=True)
SRC=Path("/kaggle/working/l2_src");SRC.mkdir(exist_ok=True)
(SRC/"knee.py").write_text(KNEE_SOURCE)
(SRC/"train_student_l2.py").write_text(HELPER_SOURCE)
(SRC/"train_student_l2_tpu.py").write_text(TPU_SOURCE)
sys.path.insert(0,str(SRC))
from train_student_l2 import load_training_tables
from knee import LABELS
for fold in range(5):
    _,_,_,training,val,gold,_=load_training_tables(A1,L2,COMP,LABELS,fold)
    assert (R2/f"student_r2_convnext_tiny_f{fold}.pt").is_file()
    print("TPU_PREFLIGHT",fold,len(training),len(val),len(gold),flush=True)


In [ ]:
command=[sys.executable,str(SRC/"train_student_l2_tpu.py"),"--a1",str(A1),"--l2",str(L2),"--r2",str(R2),"--competition",str(COMP),"--src",str(SRC),"--folds","0,1,2,3,4","--epochs","4","--batch-size","1"]
subprocess.run(command,check=True)
manifest=json.loads(Path("/kaggle/working/l2_training_manifest.json").read_text())
assert manifest["five_folds_complete"] and len(manifest["receipts"])==5
print("TPU_FIVEFOLD_CHECK_COMPLETE",json.dumps(manifest),flush=True)
